# SINdex reproduction + SINdex++ experiments (Google Colab, GPU)

Base paper: Abdaljalil et al., *SINdex: Semantic INconsistency Index for Hallucination Detection in LLMs*, IEEE OJCS 2026.

**What this notebook does:** pick a model and a dataset, then run `generate → score → label → evaluate`. Results are saved to **Google Drive**, so if Colab disconnects you re-run the cells and generation **resumes where it stopped**.

**Before you start**
1. `Runtime → Change runtime type → GPU`. Best: **A100** or **L4** (Colab Pro). A free **T4** works only with 4-bit (AWQ) models; see the *T4* preset below.
2. Upload `sindex_research.zip` to the top level of *My Drive* (or use the GitHub option in step 2).
3. For Llama models: request access on the model's Hugging Face page, and create a *read* token at huggingface.co/settings/tokens.

Run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
RUNS_DIR   = f'{DRIVE_ROOT}/sindex_runs'      # all results are stored here (persist across sessions)
os.makedirs(RUNS_DIR, exist_ok=True)

CODE_SOURCE = 'zip'   # 'zip' = sindex_research.zip in My Drive, 'git' = clone from GitHub
GIT_URL = ''          # e.g. 'https://github.com/<you>/sindex_research.git' (private repo: use a token URL)

%cd /content
if CODE_SOURCE == 'zip':
    !rm -rf /content/sindex_research && unzip -q -o "{DRIVE_ROOT}/sindex_research.zip" -d /content
else:
    !rm -rf /content/sindex_research && git clone -q {GIT_URL} /content/sindex_research
%cd /content/sindex_research
!ls

## 3. Install
`vllm` installs its own matching PyTorch. If Colab shows a *"restart session"* button after this cell, click it, then run section 2 again and continue from section 4.

In [ ]:
!pip install -q -e ".[models,dev]"
!pip install -q vllm
import importlib.metadata as md
for p in ['torch', 'vllm', 'transformers', 'sentence-transformers', 'datasets']:
    try: print(p, md.version(p))
    except Exception as e: print(p, 'NOT INSTALLED', e)

## 4. Run the unit tests (should say `17 passed`)

In [ ]:
!python -m pytest -q

## 5. Hugging Face login (needed for Llama / Gemma; you paste your own token)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 6. Choose the experiment
Pick a **preset**, or set `PRESET = 'custom'` and edit the fields.

| Preset | GPU | Purpose |
|---|---|---|
| `repro_llama2` | A100 / L4 | **Step 1: reproduce the paper** (Llama-2-7b-chat, paper's embedding labels) |
| `llama31` | A100 / L4 | Modern model, all labels, all variants |
| `qwen25` | A100 / L4 | Second modern model, no access request needed |
| `t4_qwen_awq` | free T4 | 4-bit Qwen2.5-7B, for when you only have a T4 |
| `quick_test` | any | 20 questions, P=5: check everything works in ~5 min |

Paper setting: 400 questions per dataset, P = 10 samples.

In [ ]:
PRESET  = 'quick_test'     # repro_llama2 | llama31 | qwen25 | t4_qwen_awq | quick_test | custom
DATASET = 'triviaqa'       # triviaqa | nq | squad | bioasq | jsonl
N_QUESTIONS = 400
DATA_PATH = ''             # for bioasq / jsonl: path in Drive, e.g. f'{DRIVE_ROOT}/training11b.json'
USE_LLM_JUDGE = False      # adds an LLM-judge label (slower; loads a 2nd model in the label stage)

PRESETS = {
  'repro_llama2': dict(model='meta-llama/Llama-2-7b-chat-hf', quantization=None, dtype='auto', P=10),
  'llama31':      dict(model='meta-llama/Llama-3.1-8B-Instruct', quantization=None, dtype='auto', P=10),
  'qwen25':       dict(model='Qwen/Qwen2.5-7B-Instruct', quantization=None, dtype='auto', P=10),
  't4_qwen_awq':  dict(model='Qwen/Qwen2.5-7B-Instruct-AWQ', quantization='awq', dtype='half', P=10),
  'quick_test':   dict(model='Qwen/Qwen2.5-1.5B-Instruct', quantization=None, dtype='auto', P=5),
  'custom':       dict(model='Qwen/Qwen2.5-7B-Instruct', quantization=None, dtype='auto', P=10),
}
p = PRESETS[PRESET]
if PRESET == 'quick_test':
    N_QUESTIONS = 20
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
is_t4 = 'T4' in gpu_name[0]
if is_t4 and p['quantization'] is None and PRESET != 'quick_test':
    print('WARNING: a 7-8B fp16 model does not fit on a T4. Use PRESET = "t4_qwen_awq".')
if is_t4:
    p['dtype'] = 'half'

model_tag = p['model'].split('/')[-1]
RUN_NAME = f'{model_tag}_{DATASET}_n{N_QUESTIONS}_P{p["P"]}'
OUT = f'{RUNS_DIR}/{RUN_NAME}'
print('Run name:', RUN_NAME, '\nResults ->', OUT)

## 7. Write the config (saved next to the results, so every run is reproducible)

In [ ]:
import yaml, os
os.makedirs(OUT, exist_ok=True)
paper_labels_only = PRESET == 'repro_llama2'

label_methods = ['embedding'] if paper_labels_only else ['ensemble', 'embedding', 'em', 'f1']
if USE_LLM_JUDGE:
    label_methods.append('judge')

cfg = {
  'seed': 0,
  'output_dir': OUT,
  'data': {'name': DATASET, 'n': N_QUESTIONS, **({'path': DATA_PATH} if DATA_PATH else {})},
  'generation': {
    'backend': 'vllm', 'model': p['model'], 'quantization': p['quantization'], 'dtype': p['dtype'],
    'num_samples': p['P'], 'temperature': 1.0, 'low_temperature': 0.1, 'max_new_tokens': 64,
    'batch_size': 32, 'gpu_memory_utilization': 0.85, 'max_model_len': 2048, 'use_context': True,
  },
  'scoring': {
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'device': 'cuda',
    'distance_threshold': 0.05, 'string_mode': 'question_sep', 'singleton': 'one', 'lam': 1.0,
    'variants': [
      {'name': 'adaptive', 'adaptive': True},
      {'name': 'rules', 'contradiction': 'rules'},
      {'name': 'nli', 'contradiction': 'nli'},
      {'name': 'adaptive_nli', 'adaptive': True, 'contradiction': 'nli'},
    ],
    'early_stop': {'min_samples': 3, 'patience': 2, 'tol': 0.05},
  },
  'labels': {
    'methods': label_methods, 'f1_threshold': 0.5,
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_threshold': 0.95,
    'device': 'cuda',
    'judge_backend': 'vllm',
    'judge_model': 'Qwen/Qwen2.5-7B-Instruct-AWQ' if is_t4 else 'Qwen/Qwen2.5-7B-Instruct',
    'judge_quantization': 'awq' if is_t4 else None,
    'judge_dtype': 'half' if is_t4 else 'auto',
  },
  'evaluation': {
    'n_boot': 2000, 'compare_against': 'sindex',
    'compare': ['sindex_pp', 'cluster_entropy', 'adaptive:sindex_pp', 'nli:sindex_pp',
                'adaptive_nli:sindex_pp', 'early_stop:sindex_pp', 'lin_eigv'],
  },
}
CFG_PATH = f'{OUT}/config.yaml'
with open(CFG_PATH, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(CFG_PATH).read())

## 8. Generate answers (the slow step)
Rough time on an A100 for 400 questions × 11 generations: ~5–15 min for a 7-8B model. **Resumable**: if Colab disconnects, run sections 2, 6, 7 and this cell again.

In [ ]:
!python -m hallu.pipeline --config "{CFG_PATH}" --stage generate

Look at a few answers (sanity check: are they short and on-topic?)

In [ ]:
import json, itertools
with open(f'{OUT}/generations.jsonl') as f:
    for line in itertools.islice(f, 3):
        r = json.loads(line)
        print('Q:', r['question'], '| gold:', r['answers'][:3])
        print('  most likely:', r['most_likely']['text'])
        for s in r['samples'][:5]:
            print('   -', s['text'])
        print()

## 9. Score (clustering + SINdex / SINdex++ / baselines)

In [ ]:
!python -m hallu.pipeline --config "{CFG_PATH}" --stage score

## 10. Label (which most-likely answers are hallucinations)

In [ ]:
!python -m hallu.pipeline --config "{CFG_PATH}" --stage label

## 11. Evaluate (AUROC with 95% bootstrap CI, paired tests vs SINdex)

In [ ]:
!python -m hallu.pipeline --config "{CFG_PATH}" --stage evaluate

## 12. Results table and plot

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
rep = json.load(open(f'{OUT}/report.json'))
tables = {}
for lab, res in rep['results'].items():
    df = pd.DataFrame(res['scores']).T[['auroc', 'lo', 'hi', 'aurac', 'acc@80%']].astype(float)
    df = df.sort_values('auroc', ascending=False)
    tables[lab] = df
    print(f"\n=== labels: {lab}   hallucination rate = {res['hallucination_rate']:.3f}")
    display(df.round(3))
    pv = pd.DataFrame(res.get('paired_vs_sindex', {})).T
    if len(pv):
        print('Paired bootstrap vs original SINdex (p < 0.05 = significant improvement):')
        display(pv.round(4))
if 'early_stop_mean_samples' in rep:
    print(f"\nEarly stopping used {rep['early_stop_mean_samples']:.2f} of {p['P']} samples on average")

main = 'ensemble' if 'ensemble' in tables else next(iter(tables))
df = tables[main]
fig, ax = plt.subplots(figsize=(8, 0.35 * len(df) + 1))
ax.barh(df.index, df['auroc'], xerr=[df['auroc'] - df['lo'], df['hi'] - df['auroc']], color='#4C72B0')
ax.invert_yaxis(); ax.set_xlabel('AUROC'); ax.set_xlim(0.4, 1.0)
ax.set_title(f'{RUN_NAME} (labels: {main})')
plt.tight_layout(); plt.savefig(f'{OUT}/auroc.png', dpi=200); plt.show()

## 13. Label check: paper's embedding labels vs gold-based labels
Rows where the two disagree are evidence for limitation #9 (strict cosine > 0.95 labels). Use some of these as examples in your paper.

In [ ]:
labs = pd.read_json(f'{OUT}/labels.jsonl', lines=True)
if {'embedding', 'ensemble'} <= set(labs.columns):
    dis = labs[labs['embedding'] != labs['ensemble']]
    print(f'Disagreement: {len(dis)}/{len(labs)} = {len(dis)/len(labs):.1%}')
    pd.set_option('display.max_colwidth', 120)
    display(dis[['pred', 'gold', 'embedding', 'ensemble']].head(15))

## 14. How often are answers singleton clusters? (finding #3)
If most answers sit alone at the 0.95 threshold, SINdex is close to log(P) for most questions. That supports the adaptive threshold (N4).

In [ ]:
sc = pd.read_json(f'{OUT}/scores.jsonl', lines=True)
print('mean singletons per question:', sc['n_singletons'].mean().round(2), 'of', p['P'])
print('mean clusters (fixed 0.95):  ', sc['num_clusters'].mean().round(2))
print('mean clusters (adaptive):    ', sc['adaptive:num_clusters'].mean().round(2))
sc[['num_clusters', 'adaptive:num_clusters']].plot.hist(bins=range(1, p['P'] + 2), alpha=0.6)
plt.xlabel('clusters per question'); plt.show()

## 15. Run the full grid (optional, long)
Runs every (model, dataset) pair one after another. Each run is resumable, so you can stop and restart. Do **one row of Table 1** at a time if your Colab session is short.

In [ ]:
GRID_MODELS   = ['meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']
GRID_DATASETS = ['triviaqa', 'nq', 'squad']
RUN_GRID = False   # set True to start

import copy, subprocess
if RUN_GRID:
    for m in GRID_MODELS:
        for d in GRID_DATASETS:
            c = copy.deepcopy(cfg)
            name = f"{m.split('/')[-1]}_{d}_n{N_QUESTIONS}_P{p['P']}"
            c['output_dir'] = f'{RUNS_DIR}/{name}'
            c['data'] = {'name': d, 'n': N_QUESTIONS}
            c['generation']['model'] = m
            os.makedirs(c['output_dir'], exist_ok=True)
            path = f"{c['output_dir']}/config.yaml"
            yaml.safe_dump(c, open(path, 'w'), sort_keys=False)
            for stage in ['generate', 'score', 'label', 'evaluate']:
                print(f'>>> {name}: {stage}')
                subprocess.run(['python', '-m', 'hallu.pipeline', '--config', path, '--stage', stage], check=True)

## 16. Summary across all runs in Drive (copy into your paper's Table 1)

In [ ]:
import glob
rows = []
for rp in sorted(glob.glob(f'{RUNS_DIR}/*/report.json')):
    r = json.load(open(rp)); run = rp.split('/')[-2]
    for lab, res in r['results'].items():
        for score in ['sindex', 'sindex_pp', 'adaptive:sindex_pp', 'nli:sindex_pp', 'cluster_entropy', 'lin_eigv', 'naive_entropy']:
            if score in res['scores']:
                rows.append({'run': run, 'labels': lab, 'score': score, 'auroc': res['scores'][score]['auroc']})
summary = pd.DataFrame(rows)
if len(summary):
    pivot = summary.pivot_table(index=['run', 'labels'], columns='score', values='auroc').round(3)
    display(pivot)
    pivot.to_csv(f'{RUNS_DIR}/summary_auroc.csv')
    print('saved', f'{RUNS_DIR}/summary_auroc.csv')

---
**Reproduction target (paper, Table 1):** SINdex beats semantic entropy by up to 7.6% (TriviaQA), 9.3% (NQ), 9.1% (SQuAD) and 4.8% (BioASQ) AUROC. With `repro_llama2`, your `sindex` AUROC should be within about ±0.02 of their reported value. If it isn't, check the prompt, temperature and label settings before moving on.

**Rules for honest results:** tune `lam`, the thresholds and the early-stop settings on a *separate* set of questions (e.g. `seed: 1`), then report on the test questions (`seed: 0`).